# H&M NGCF seed49 — 새 GPU 전용 M5_A~D
기존 M1~M4 Pod는 유지합니다. 기존 H&M Network Volume을 새 Pod의 `/workspace`에도 연결하세요. 새 학습은 별도 폴더에 저장됩니다.


## 1. 기존 설치·분담·입력·GPU 점검
재설치/입력 복사 없이 기존 환경을 읽어 재사용합니다. 이 셀에서는 학습하지 않습니다.


In [ ]:
from pathlib import Path
import subprocess, datetime, json, os
PACKAGE=Path('/workspace/hm_ngcf_m5_start')
PYTHON=Path('/workspace/clv/venv/bin/python')
ROOT=Path('/workspace/clv_hm_ngcf_m5_seed49')
assert (PACKAGE/'hm_ngcf_m5.py').is_file(), '패키지를 /workspace에 압축 해제하세요.'
assert PYTHON.is_file(), '기존 H&M 영구볼륨/환경이 없습니다. 기존 서버에 재설치하지 말고 연결부터 확인하세요.'
BASE=[str(PYTHON), '-u', str(PACKAGE/'hm_ngcf_m5.py')]
subprocess.run(BASE+['check'], check=True)


## 2. M5_A → B → C → D 시작
공유볼륨의 M1~M4 배정은 코드가 검사합니다. 다른 볼륨·Colab에 동일 M5 seed49가 없는지 확인하고 True로 바꾸세요. 모형·시드는 고정이며 기존 실행에는 손대지 않습니다.


In [ ]:
CONFIRMED_NO_DUPLICATE=False
assert CONFIRMED_NO_DUPLICATE, '다른 곳의 동일 H&M NGCF seed49 M5 실행이 없는지 확인하세요.'
logs=ROOT/'logs'; logs.mkdir(parents=True, exist_ok=True)
LOG=logs/('hm-ngcf-m5_'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S_%f')+'.log')
with LOG.open('x') as output:
    process=subprocess.Popen(BASE+['run','--confirmed-unstarted'], stdout=output,
        stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)
print('시작 요청 PID:', process.pid, '\n로그:', LOG)
print('아래에서 첫 배치/epoch 저장을 확인하세요. 프로세스 종료 후에도 Pod 과금은 계속됩니다.')


## 3. 현재 진행·학습/저장 시간 확인
요청할 때만 로그를 읽습니다. 원래 Pod의 진행 파일은 변경하지 않습니다.


In [ ]:
paths=sorted((ROOT/'logs').glob('hm-ngcf-m5_*.log'))
assert paths, '시작 로그가 없습니다.'
print(paths[-1])
print('\n'.join(paths[-1].read_text(errors='replace').splitlines()[-35:]))
for p in sorted((ROOT/'results/hm-ngcf').glob('*/progress/progress.json')):
    print(p, p.read_text())
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.used,utilization.gpu','--format=csv'],check=True)


## 4. 결과
완료한 모형별 결과는 `/workspace/clv_hm_ngcf_m5_seed49/exports/hm-ngcf_results_partial.zip`에 저장됩니다. 네 혼합모형을 모두 끝내도 전체 8모형 기준으로는 부분 ZIP입니다. 최신·직전 재개 파일과 epoch별 시간 기록은 새 root의 `results/hm-ngcf`에 남습니다. 기존 M1~M4와 통합할 때는 원본 신원을 별도 검증합니다. 자동 Pod 정지는 없습니다.
